# **Multi-Model Ensemble PRCPTOT and Water Stress Analyses**
## Three GCMs: HadGEM2-AO, MPI-ESM-MR, IPSL-CM5A-LR
## RCP 4.5 Scenario - Indonesia and Russia

**Analysis Period:**
- Baseline: 1976-2005 (30 years)
- Future: 2041-2070 (30 years)

**Models:**
1. HadGEM2-AO (UK Met Office) - 360-day calendar
2. MPI-ESM-MR (Max Planck Institute) - Standard calendar
3. IPSL-CM5A-LR (Institut Pierre-Simon Laplace) - 360-day calendar

## Step 1: Import Libraries

In [ ]:
import xarray as xr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import os
import shutil
from glob import glob
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi']       = 100
plt.rcParams['savefig.dpi']      = 300
plt.rcParams['savefig.bbox']     = 'tight'
plt.rcParams['font.size']        = 10
plt.rcParams['axes.titlesize']   = 12
plt.rcParams['axes.labelsize']   = 11
plt.rcParams['xtick.labelsize']  = 9
plt.rcParams['ytick.labelsize']  = 9
plt.rcParams['legend.fontsize']  = 10
plt.rcParams['figure.titlesize'] = 13
plt.rcParams['contour.negative_linestyle'] = 'solid'
%matplotlib inline

print("Libraries imported successfully")
print("300 DPI + smooth-gradient plotting configured")


## Step 2: File Paths and Models

In [ ]:
# Base directory
BASE_DIR = r"D:\Tugas\Personal\Jupyter Notebook\ESGF"

# Define models
models = {
    'HadGEM2-AO': {
        'historical': os.path.join(BASE_DIR, 'pr_day_HadGEM2-AO_historical_r1i1p1_18600101-20051230.nc'),
        'rcp45': os.path.join(BASE_DIR, 'pr_day_HadGEM2-AO_rcp45_r1i1p1_20060101-21001230.nc'),
        'calendar': '360_day'
    },
    'MPI-ESM-MR': {
        'historical': [  # Merge multiple files
            os.path.join(BASE_DIR, 'pr_day_MPI-ESM-MR_historical_r1i1p1_19700101-19791231.nc'),
            os.path.join(BASE_DIR, 'pr_day_MPI-ESM-MR_historical_r1i1p1_19800101-19891231.nc'),
            os.path.join(BASE_DIR, 'pr_day_MPI-ESM-MR_historical_r1i1p1_19900101-19991231.nc'),
            os.path.join(BASE_DIR, 'pr_day_MPI-ESM-MR_historical_r1i1p1_20000101-20051231.nc')
        ],
        'rcp45': [  # Merge multiple files
            os.path.join(BASE_DIR, 'pr_day_MPI-ESM-MR_rcp45_r1i1p1_20400101-20491231.nc'),
            os.path.join(BASE_DIR, 'pr_day_MPI-ESM-MR_rcp45_r1i1p1_20500101-20591231.nc'),
            os.path.join(BASE_DIR, 'pr_day_MPI-ESM-MR_rcp45_r1i1p1_20600101-20691231.nc'),
            os.path.join(BASE_DIR, 'pr_day_MPI-ESM-MR_rcp45_r1i1p1_20700101-20791231.nc')
        ],
        'calendar': 'standard'
    },
    'IPSL-CM5A-LR': {
        'historical': os.path.join(BASE_DIR, 'pr_day_IPSL-CM5A-LR_historical_r1i1p1_19500101-20051231.nc'),
        'rcp45': os.path.join(BASE_DIR, 'pr_day_IPSL-CM5A-LR_rcp45_r1i1p1_20060101-22051231.nc'),
        'calendar': '360_day'
    }
}

# Regional boundaries
indonesia_bounds = {'lon': slice(95, 141), 'lat': slice(-11, 6)}
russia_bounds = {'lon': slice(19, 192), 'lat': slice(41, 82)}

print("File paths configured!")
print(f"\nModels to process: {list(models.keys())}")

## STEP 3: Visualization settings

In [ ]:
import matplotlib.colors as mcolors
import numpy as np
from scipy.ndimage import zoom as _ndz

# ===== 10m ocean/lake mask  ====================
_OCEAN_FEAT = cfeature.NaturalEarthFeature(
    'physical', 'ocean', '10m', facecolor='white', edgecolor='none')
_LAKES_FEAT = cfeature.NaturalEarthFeature(
    'physical', 'lakes', '10m', facecolor='white', edgecolor='none')

def _add_ocean_mask(ax):
    ax.add_feature(_OCEAN_FEAT, zorder=2)
    ax.add_feature(_LAKES_FEAT, zorder=2)

def _upsample_bilinear(da, factor=8):
    """Bilinear upsample.  sortby('lat') first so CHIRPS descending-lat grids are flipped to ascending before linspace/pcolormesh sees them."""
    da  = da.sortby('lat')
    lons = da.lon.values.astype(float)
    lats = da.lat.values.astype(float)
    data = da.values.astype(float)
    d_up = _ndz(data, factor, order=1)
    l_up = np.linspace(lons[0], lons[-1], d_up.shape[1])
    a_up = np.linspace(lats[0], lats[-1], d_up.shape[0])
    return l_up, a_up, d_up

def _upsample_nearest(da, factor=8):
    """Nearest-neighbour upsample for categorical data (keeps integers exact)."""
    da  = da.sortby('lat')
    lons = da.lon.values.astype(float)
    lats = da.lat.values.astype(float)
    data = da.values.astype(float)
    d_up = _ndz(data, factor, order=0)
    l_up = np.linspace(lons[0], lons[-1], d_up.shape[1])
    a_up = np.linspace(lats[0], lats[-1], d_up.shape[0])
    return l_up, a_up, d_up

def _add_style(ax, im, unit, pad, shrink, ticks, label_size, extent=None):
    """Ocean mask → coastlines → borders → gridlines → colorbar."""
    _add_ocean_mask(ax)
    ax.coastlines(resolution='10m', linewidth=0.8, color='black', zorder=3)
    ax.add_feature(cfeature.BORDERS, linestyle=':', linewidth=0.6,
                   alpha=0.8, zorder=3)
    if extent is not None:
        ax.set_extent(extent, crs=ccrs.PlateCarree())
    gl = ax.gridlines(draw_labels=True, linestyle='--',
                      linewidth=0.4, alpha=0.5, color='gray', zorder=4)
    gl.top_labels   = False
    gl.right_labels = False
    gl.xlabel_style = {'size': label_size - 1}
    gl.ylabel_style = {'size': label_size - 1}
    cbar = plt.colorbar(im, ax=ax, pad=pad, shrink=shrink, ticks=ticks)
    cbar.ax.yaxis.set_major_formatter(
        plt.FuncFormatter(lambda x, _: f'{int(x)}'))
    cbar.ax.tick_params(labelsize=label_size - 1)
    return cbar

def _pcolormesh_diverging(ax, da, vmin, vmax, cmap='RdBu_r',
                           center=0, pad=0.10, shrink=0.72,
                           unit='mm/year', tick_step=None,
                           label_size=9, extent=None):
    norm = mcolors.TwoSlopeNorm(vmin=vmin, vcenter=center, vmax=vmax)
    lons_f, lats_f, data_f = _upsample_bilinear(da, factor=8)
    im = ax.pcolormesh(lons_f, lats_f, data_f,
                       cmap=cmap, norm=norm,
                       transform=ccrs.PlateCarree(),
                       shading='gouraud', zorder=1)
    if tick_step is None:
        span = vmax - vmin
        tick_step = next((s for s in [200,100,50,25,10] if span/s <= 12), 50)
    ticks = list(range(int(vmin), int(vmax)+1, tick_step))
    cbar  = _add_style(ax, im, unit, pad, shrink, ticks, label_size, extent)
    cbar.set_label(unit, size=label_size)
    return im

def _pcolormesh_sequential(ax, da, vmin, vmax, cmap='YlOrRd',
                            pad=0.10, shrink=0.72,
                            unit='mm/year', tick_step=None,
                            label_size=9, extent=None):
    norm = mcolors.Normalize(vmin=vmin, vmax=vmax)
    lons_f, lats_f, data_f = _upsample_bilinear(da, factor=8)
    im = ax.pcolormesh(lons_f, lats_f, data_f,
                       cmap=cmap, norm=norm,
                       transform=ccrs.PlateCarree(),
                       shading='gouraud', zorder=1)
    if tick_step is None:
        span = vmax - vmin
        tick_step = next((s for s in [100,50,25,10] if span/s <= 10), 25)
    ticks = list(range(int(vmin), int(vmax)+1, tick_step))
    cbar  = _add_style(ax, im, unit, pad, shrink, ticks, label_size, extent)
    cbar.set_label(unit, size=label_size)
    return im

def _apply_water_stress_style(ax, da, pad=0.10, shrink=0.72,
                               label_size=9, extent=None):
    """Plot CWSI stress categories.
    Category 1 = Low stress    (WSI >= -10%)
    Category 2 = Moderate      (-20% <= WSI < -10%)
    Category 3 = High stress   (WSI < -20%)

    Land-coverage fix: interp to a fine uniform grid before upsampling so every land pixel gets a category value (no white gaps).
    Single colorbar: _add_style() is skipped; we build the colorbar here directly to avoid the duplicate that _add_style() would create.
    """
    cmap_ws = mcolors.ListedColormap(['#4CAF50', '#FFC107', '#F44336'])
    norm_ws = mcolors.BoundaryNorm([0.5, 1.5, 2.5, 3.5], ncolors=3)

    # ===== Fill land-coverage gaps before upsampling ====================
    da_s = da.sortby('lat')
    fine_lons = np.arange(float(da_s.lon.min()), float(da_s.lon.max()) + 0.25, 0.25)
    fine_lats = np.arange(float(da_s.lat.min()), float(da_s.lat.max()) + 0.25, 0.25)
    da_fine = da_s.interp(lon=fine_lons, lat=fine_lats, method='nearest')

    lons_f, lats_f, data_f = _upsample_nearest(da_fine, factor=4)

    im = ax.pcolormesh(lons_f, lats_f, data_f,
                       cmap=cmap_ws, norm=norm_ws,
                       transform=ccrs.PlateCarree(),
                       shading='nearest', zorder=1)

    # ===== Ocean mask, coastlines, borders, gridlines ====================
    _add_ocean_mask(ax)
    ax.coastlines(resolution='10m', linewidth=0.8, color='black', zorder=3)
    ax.add_feature(cfeature.BORDERS, linestyle=':', linewidth=0.6,
                   alpha=0.8, zorder=3)
    if extent is not None:
        ax.set_extent(extent, crs=ccrs.PlateCarree())
    gl = ax.gridlines(draw_labels=True, linestyle='--',
                      linewidth=0.4, alpha=0.5, color='gray', zorder=4)
    gl.top_labels   = False
    gl.right_labels = False
    gl.xlabel_style = {'size': label_size - 1}
    gl.ylabel_style = {'size': label_size - 1}

    # ===== Single colorbar ====================
    cbar = plt.colorbar(im, ax=ax, pad=pad, shrink=shrink, ticks=[1, 2, 3])
    cbar.ax.set_yticklabels(
        ['Low\n(WSI ≥ -10%)',
         'Moderate\n(-20% ≤ WSI < -10%)',
         'High\n(WSI < -20%)'],
        fontsize=label_size - 1)
    return im

print('Helpers ready (factor=8, sortby lat, 10m ocean mask, CWSI labels)')


## Step 4: Load and Process Each Model

In [ ]:
def load_and_merge_files(file_paths):
    if isinstance(file_paths, str):
        return xr.open_dataset(file_paths)
    else:
        # Multiple files - merge them
        datasets = [xr.open_dataset(f) for f in file_paths]
        return xr.concat(datasets, dim='time')

def adjust_calendar_dates(calendar_type, start_year, end_year):
    if calendar_type == '360_day':
        return f"{start_year}-01-01", f"{end_year}-12-30"
    else:  # standard
        return f"{start_year}-01-01", f"{end_year}-12-31"

# Storage for processed data
model_data = {}

print("="*55)
print("LOADING AND PROCESSING MODELS")
print("="*55)

for model_name, model_info in models.items():
    print(f"Processing: {model_name}")
    print(f"Calendar: {model_info['calendar']}")
    
    try:
        # Load historical data
        print("\n1. Loading historical data...")
        ds_hist = load_and_merge_files(model_info['historical'])
        ds_hist['pr_mm'] = ds_hist['pr'] * 86400  # Convert to mm/day
        
        # Select baseline period (1976-2005)
        baseline_start, baseline_end = adjust_calendar_dates(model_info['calendar'], 1976, 2005)
        ds_baseline = ds_hist.sel(time=slice(baseline_start, baseline_end))
        print(f"   Baseline: {ds_baseline.time.values[0]} to {ds_baseline.time.values[-1]}")
        print(f"   Days: {len(ds_baseline.time)}")
        
        # Load RCP 4.5 data
        print("\n2. Loading RCP 4.5 data...")
        ds_rcp = load_and_merge_files(model_info['rcp45'])
        ds_rcp['pr_mm'] = ds_rcp['pr'] * 86400  # Convert to mm/day
        
        # Select future period (2041-2070)
        future_start, future_end = adjust_calendar_dates(model_info['calendar'], 2041, 2070)
        ds_future = ds_rcp.sel(time=slice(future_start, future_end))
        print(f"   Future: {ds_future.time.values[0]} to {ds_future.time.values[-1]}")
        print(f"   Days: {len(ds_future.time)}")
        
        # Calculate annual PRCPTOT
        print("\n3. Calculating PRCPTOT...")
        prcptot_baseline = ds_baseline['pr_mm'].resample(time='YE').sum()
        prcptot_future = ds_future['pr_mm'].resample(time='YE').sum()
        
        # Calculate means
        baseline_mean = prcptot_baseline.mean(dim='time')
        future_mean = prcptot_future.mean(dim='time')
        anomaly = future_mean - baseline_mean
        
        # Store results
        model_data[model_name] = {
            'baseline_mean': baseline_mean,
            'future_mean': future_mean,
            'anomaly': anomaly,
            'prcptot_baseline': prcptot_baseline,
            'prcptot_future': prcptot_future
        }
        
        print(f"\n{model_name} processed successfully!")
        print(f"     Global baseline mean: {float(baseline_mean.mean()):.2f} mm/year")
        print(f"     Global future mean: {float(future_mean.mean()):.2f} mm/year")
        print(f"     Global anomaly: {float(anomaly.mean()):.2f} mm/year")
        
    except Exception as e:
        print(f"\nError processing {model_name}: {e}")
        continue

print(f"\n{'='*55}")
print(f"Successfully processed {len(model_data)} models")
print(f"{'='*55}")

## Step 4.5: Debug Loaded Models

In [ ]:
# Debugging: Check what was loaded
print("="*55)
print("DEBUGGING: CHECKING LOADED MODELS")
print("="*55)

print(f"\nNumber of models loaded: {len(model_data)}")
print(f"Models: {list(model_data.keys())}")

if len(model_data) == 0:
    print("\nWARNING: NO MODELS WERE LOADED!")
    print("\nPlease check the error messages in Step 3 above.")
    print("\nExpected files:")
    for model_name, info in models.items():
        print(f"\n{model_name}:")
        if isinstance(info['historical'], list):
            for f in info['historical']:
                print(f"  Historical: {os.path.exists(f)} - {f}")
        else:
            print(f"  Historical: {os.path.exists(info['historical'])} - {info['historical']}")
        
        if isinstance(info['rcp45'], list):
            for f in info['rcp45']:
                print(f"  RCP45: {os.path.exists(f)} - {f}")
        else:
            print(f"  RCP45: {os.path.exists(info['rcp45'])} - {info['rcp45']}")
else:
    print("\n  Models loaded successfully!")
    for model_name in model_data.keys():
        print(f"\n{model_name}:")
        print(f"  Baseline shape: {model_data[model_name]['baseline_mean'].shape}")
        print(f"  Future shape: {model_data[model_name]['future_mean'].shape}")
        print(f"  Anomaly shape: {model_data[model_name]['anomaly'].shape}")

## Step 5: Calculate Multi-Model Ensemble Mean

In [ ]:
print("="*55)
print("CALCULATING MULTI-MODEL ENSEMBLE MEAN")
print("="*55)

# Check which models were successfully loaded
print(f"\nModels successfully loaded: {list(model_data.keys())}")
print(f"Number of models: {len(model_data)}")

if len(model_data) == 0:
    print("\n  ERROR: No models were loaded successfully!")
    print("Please check:")
    print("  1. File paths in Step 2 are correct")
    print("  2. Files exist in the specified location")
    print("  3. Check error messages in Step 3 above")
    raise Exception("No models loaded. Cannot proceed with ensemble analysis.")

# Use first available model as reference grid
reference_model = list(model_data.keys())[0]
reference_grid = model_data[reference_model]['baseline_mean']
print(f"\nUsing {reference_model} as reference grid ({len(reference_grid.lat)}x{len(reference_grid.lon)})")

# Regrid all models to reference grid
regridded_data = {}

for model_name, data in model_data.items():
    print(f"\nRegridding {model_name}...")
    
    # Regrid to reference grid using nearest neighbor
    baseline_regrid = data['baseline_mean'].interp(
        lat=reference_grid.lat,
        lon=reference_grid.lon,
        method='nearest'
    )
    
    future_regrid = data['future_mean'].interp(
        lat=reference_grid.lat,
        lon=reference_grid.lon,
        method='nearest'
    )
    
    anomaly_regrid = data['anomaly'].interp(
        lat=reference_grid.lat,
        lon=reference_grid.lon,
        method='nearest'
    )
    
    regridded_data[model_name] = {
        'baseline': baseline_regrid,
        'future': future_regrid,
        'anomaly': anomaly_regrid
    }
    
    print(f"     {model_name} regridded to {len(reference_grid.lat)}x{len(reference_grid.lon)} grid")

# Calculate ensemble mean
print("\nCalculating ensemble statistics...")

# Stack all models
baseline_stack = xr.concat([regridded_data[m]['baseline'] for m in regridded_data.keys()], dim='model')
future_stack = xr.concat([regridded_data[m]['future'] for m in regridded_data.keys()], dim='model')
anomaly_stack = xr.concat([regridded_data[m]['anomaly'] for m in regridded_data.keys()], dim='model')

# Calculate ensemble mean and standard deviation
ensemble_baseline_mean = baseline_stack.mean(dim='model')
ensemble_baseline_std = baseline_stack.std(dim='model')

ensemble_future_mean = future_stack.mean(dim='model')
ensemble_future_std = future_stack.std(dim='model')

ensemble_anomaly_mean = anomaly_stack.mean(dim='model')
ensemble_anomaly_std = anomaly_stack.std(dim='model')

print("\n" + "="*55)
print("ENSEMBLE STATISTICS (Global)")
print("="*55)
print(f"\nBaseline (1976-2005):")
print(f"   Ensemble Mean: {float(ensemble_baseline_mean.mean()):.2f} mm/year")
print(f"   Ensemble Std:  {float(ensemble_baseline_std.mean()):.2f} mm/year")

print(f"\nFuture (2041-2070):")
print(f"   Ensemble Mean: {float(ensemble_future_mean.mean()):.2f} mm/year")
print(f"   Ensemble Std:  {float(ensemble_future_std.mean()):.2f} mm/year")

print(f"\nAnomaly (Future - Baseline):")
print(f"   Ensemble Mean: {float(ensemble_anomaly_mean.mean()):.2f} mm/year")
print(f"   Ensemble Std:  {float(ensemble_anomaly_std.mean()):.2f} mm/year")

print("\nMulti-model ensemble calculated")

## Step 6: Extract Regional Data

In [ ]:
# Extract for ensemble mean
ensemble_indonesia = {
    'baseline': ensemble_baseline_mean.sel(**indonesia_bounds),
    'future': ensemble_future_mean.sel(**indonesia_bounds),
    'anomaly': ensemble_anomaly_mean.sel(**indonesia_bounds),
    'std': ensemble_anomaly_std.sel(**indonesia_bounds)
}

ensemble_russia = {
    'baseline': ensemble_baseline_mean.sel(**russia_bounds),
    'future': ensemble_future_mean.sel(**russia_bounds),
    'anomaly': ensemble_anomaly_mean.sel(**russia_bounds),
    'std': ensemble_anomaly_std.sel(**russia_bounds)
}

# Extract for individual models
individual_indonesia = {}
individual_russia = {}

for model_name in regridded_data.keys():
    individual_indonesia[model_name] = {
        'baseline': regridded_data[model_name]['baseline'].sel(**indonesia_bounds),
        'future': regridded_data[model_name]['future'].sel(**indonesia_bounds),
        'anomaly': regridded_data[model_name]['anomaly'].sel(**indonesia_bounds)
    }
    
    individual_russia[model_name] = {
        'baseline': regridded_data[model_name]['baseline'].sel(**russia_bounds),
        'future': regridded_data[model_name]['future'].sel(**russia_bounds),
        'anomaly': regridded_data[model_name]['anomaly'].sel(**russia_bounds)
    }

print("INDONESIA - Ensemble Statistics:")
print(f"   Baseline: {float(ensemble_indonesia['baseline'].mean()):.2f} ± {float(ensemble_indonesia['std'].mean()):.2f} mm/year")
print(f"   Future:   {float(ensemble_indonesia['future'].mean()):.2f} mm/year")
print(f"   Anomaly:  {float(ensemble_indonesia['anomaly'].mean()):.2f} ± {float(ensemble_indonesia['std'].mean()):.2f} mm/year")

print("\nRUSSIA - Ensemble Statistics:")
print(f"   Baseline: {float(ensemble_russia['baseline'].mean()):.2f} ± {float(ensemble_russia['std'].mean()):.2f} mm/year")
print(f"   Future:   {float(ensemble_russia['future'].mean()):.2f} mm/year")
print(f"   Anomaly:  {float(ensemble_russia['anomaly'].mean()):.2f} ± {float(ensemble_russia['std'].mean()):.2f} mm/year")

print("\nRegional data extracted.")

## Step 7: Calculate Water Stress

In [ ]:
# ── Climate Water Stress Index (CWSI) ────────────────────────────────────────
# WSI = ((P_future - P_baseline) / P_baseline) * 100  [% change]
# Negative = drier (more stress), Positive = wetter (less stress)

def calculate_water_stress_index(future, baseline):
    """Relative precipitation anomaly as a percentage of the baseline."""
    wsi = ((future - baseline) / baseline) * 100
    return wsi

def classify_water_stress(wsi):
    """Classify CWSI into 3 stress categories based on % anomaly.
    Category 1 — Low stress    : WSI >= -10%  (near-normal or wetter)
    Category 2 — Moderate      : -20% <= WSI < -10%
    Category 3 — High stress   : WSI < -20%  (strongly drier)
    """
    stress = xr.zeros_like(wsi)
    stress = xr.where(wsi < -20,               3, stress)  # High
    stress = xr.where((wsi >= -20) & (wsi < -10), 2, stress)  # Moderate
    stress = xr.where(wsi >= -10,              1, stress)  # Low
    return stress

def calculate_stress_percentages(stress_data):
    total    = stress_data.size
    high     = float((stress_data == 3).sum()) / total * 100
    moderate = float((stress_data == 2).sum()) / total * 100
    low      = float((stress_data == 1).sum()) / total * 100
    return high, moderate, low

# ── Calculate CWSI for both regions ──────────────────────────────────────────
indo_wsi    = calculate_water_stress_index(
    ensemble_indonesia['future'],
    ensemble_indonesia['baseline'])

russia_wsi  = calculate_water_stress_index(
    ensemble_russia['future'],
    ensemble_russia['baseline'])

water_stress_indonesia = classify_water_stress(indo_wsi)
water_stress_russia    = classify_water_stress(russia_wsi)

indo_high,   indo_mod,   indo_low   = calculate_stress_percentages(water_stress_indonesia)
russia_high, russia_mod, russia_low = calculate_stress_percentages(water_stress_russia)

print("=" * 55)
print("CWSI WATER STRESS ASSESSMENT (Multi-Model Ensemble)")
print("=" * 55)
print("\nIndonesia (2041-2070 vs baseline):")
print(f"   High stress   (WSI < -20%):         {indo_high:.1f}%")
print(f"   Moderate      (-20% ≤ WSI < -10%):  {indo_mod:.1f}%")
print(f"   Low stress    (WSI ≥ -10%):          {indo_low:.1f}%")
print(f"   Mean WSI: {float(indo_wsi.mean()):.1f}%")
print("\nRussia (2041-2070 vs baseline):")
print(f"   High stress   (WSI < -20%):         {russia_high:.1f}%")
print(f"   Moderate      (-20% ≤ WSI < -10%):  {russia_mod:.1f}%")
print(f"   Low stress    (WSI ≥ -10%):          {russia_low:.1f}%")
print(f"   Mean WSI: {float(russia_wsi.mean()):.1f}%")


## Step 8: Create All Visualizations

In [ ]:
# FIGURE 1: Multi-Model Comparison — Indonesia
INDO_PROJ   = ccrs.PlateCarree()
INDO_EXTENT = [94, 141, -11, 6]

fig = plt.figure(figsize=(20, 12))

for i, model_name in enumerate(['HadGEM2-AO', 'MPI-ESM-MR', 'IPSL-CM5A-LR']):
    ax = fig.add_subplot(3, 3, i+1, projection=INDO_PROJ)
    _pcolormesh_diverging(ax, individual_indonesia[model_name]['anomaly'],
                          vmin=-400, vmax=400, cmap='RdBu_r', center=0,
                          tick_step=100, unit='mm/year', extent=INDO_EXTENT)
    ax.set_title(f'Indonesia – {model_name}\nAnomaly (2041–2070 vs 1976–2005)',
                 fontsize=10, fontweight='bold')

ax4 = fig.add_subplot(3, 3, 4, projection=INDO_PROJ)
_pcolormesh_diverging(ax4, ensemble_indonesia['anomaly'],
                      vmin=-400, vmax=400, cmap='RdBu_r', center=0,
                      tick_step=100, unit='mm/year', extent=INDO_EXTENT)
ax4.set_title('Indonesia – Ensemble Mean Anomaly', fontsize=10, fontweight='bold')

ax5 = fig.add_subplot(3, 3, 5, projection=INDO_PROJ)
_pcolormesh_sequential(ax5, ensemble_indonesia['std'],
                       vmin=0, vmax=300, cmap='YlOrRd',
                       tick_step=50, unit='mm/year', extent=INDO_EXTENT)
ax5.set_title('Indonesia – Model Uncertainty (Std Dev)', fontsize=10, fontweight='bold')

ax6 = fig.add_subplot(3, 3, 6, projection=INDO_PROJ)
_apply_water_stress_style(ax6, water_stress_indonesia, extent=INDO_EXTENT)
ax6.set_title('Indonesia – Water Stress (CWSI, Ensemble)', fontsize=10, fontweight='bold')

ax7 = fig.add_subplot(3, 3, 7, projection=INDO_PROJ)
positive_change = (anomaly_stack.sel(**indonesia_bounds) > 0).sum(dim='model') / len(models) * 100
_pcolormesh_diverging(ax7, positive_change,
                      vmin=0, vmax=100, cmap='RdBu_r', center=50,
                      tick_step=25, unit='% Models', extent=INDO_EXTENT)
ax7.set_title('Indonesia – % Models Showing Increase', fontsize=10, fontweight='bold')

plt.tight_layout(pad=2.0, w_pad=3.0, h_pad=3.5)
plt.savefig('MultiModel_Indonesia_Analysis.png', dpi=300,
            bbox_inches='tight', facecolor='white')
plt.show()
print('Indonesia figure saved.')


In [ ]:
# FIGURE 2: Multi-Model Comparison — Russia
RUSSIA_PROJ   = ccrs.PlateCarree(central_longitude=105)
RUSSIA_EXTENT = [19, 192, 40, 83]

fig = plt.figure(figsize=(24, 12))

for i, model_name in enumerate(['HadGEM2-AO', 'MPI-ESM-MR', 'IPSL-CM5A-LR']):
    ax = fig.add_subplot(3, 3, i+1, projection=RUSSIA_PROJ)
    _pcolormesh_diverging(ax, individual_russia[model_name]['anomaly'],
                          vmin=-160, vmax=160, cmap='RdBu_r', center=0,
                          tick_step=40, unit='mm/year', extent=RUSSIA_EXTENT)
    ax.set_title(f'Russia – {model_name}\nAnomaly (2041–2070 vs 1976–2005)',
                 fontsize=10, fontweight='bold')

ax4 = fig.add_subplot(3, 3, 4, projection=RUSSIA_PROJ)
_pcolormesh_diverging(ax4, ensemble_russia['anomaly'],
                      vmin=-160, vmax=160, cmap='RdBu_r', center=0,
                      tick_step=40, unit='mm/year', extent=RUSSIA_EXTENT)
ax4.set_title('Russia – Ensemble Mean Anomaly', fontsize=10, fontweight='bold')

ax5 = fig.add_subplot(3, 3, 5, projection=RUSSIA_PROJ)
_pcolormesh_sequential(ax5, ensemble_russia['std'],
                       vmin=0, vmax=150, cmap='YlOrRd',
                       tick_step=25, unit='mm/year', extent=RUSSIA_EXTENT)
ax5.set_title('Russia – Model Uncertainty (Std Dev)', fontsize=10, fontweight='bold')

ax6 = fig.add_subplot(3, 3, 6, projection=RUSSIA_PROJ)
_apply_water_stress_style(ax6, water_stress_russia, extent=RUSSIA_EXTENT)
ax6.set_title('Russia – Water Stress (CWSI, Ensemble)', fontsize=10, fontweight='bold')

ax7 = fig.add_subplot(3, 3, 7, projection=RUSSIA_PROJ)
positive_change = (anomaly_stack.sel(**russia_bounds) > 0).sum(dim='model') / len(models) * 100
_pcolormesh_diverging(ax7, positive_change,
                      vmin=0, vmax=100, cmap='RdBu_r', center=50,
                      tick_step=25, unit='% Models', extent=RUSSIA_EXTENT)
ax7.set_title('Russia – % Models Showing Increase', fontsize=10, fontweight='bold')

plt.tight_layout(pad=2.0, w_pad=3.0, h_pad=3.5)
plt.savefig('MultiModel_Russia_Analysis.png', dpi=300,
            bbox_inches='tight', facecolor='white')
plt.show()
print('Russia figure saved.')


## Step 9: Export Summary Statistics

In [ ]:
summary_data = []

# Add individual models - Indonesia
for model_name in ['HadGEM2-AO', 'MPI-ESM-MR', 'IPSL-CM5A-LR']:
    summary_data.append({
        'Region': 'Indonesia',
        'Model': model_name,
        'Type': 'Individual',
        'Baseline_mm': float(individual_indonesia[model_name]['baseline'].mean()),
        'Future_mm': float(individual_indonesia[model_name]['future'].mean()),
        'Anomaly_mm': float(individual_indonesia[model_name]['anomaly'].mean()),
        'Change_percent': float(individual_indonesia[model_name]['anomaly'].mean()) / 
                         float(individual_indonesia[model_name]['baseline'].mean()) * 100
    })

# Add ensemble - Indonesia
summary_data.append({
    'Region': 'Indonesia',
    'Model': 'Ensemble Mean',
    'Type': 'Ensemble',
    'Baseline_mm': float(ensemble_indonesia['baseline'].mean()),
    'Future_mm': float(ensemble_indonesia['future'].mean()),
    'Anomaly_mm': float(ensemble_indonesia['anomaly'].mean()),
    'Change_percent': float(ensemble_indonesia['anomaly'].mean()) / 
                     float(ensemble_indonesia['baseline'].mean()) * 100
})

# Add individual models - Russia
for model_name in ['HadGEM2-AO', 'MPI-ESM-MR', 'IPSL-CM5A-LR']:
    summary_data.append({
        'Region': 'Russia',
        'Model': model_name,
        'Type': 'Individual',
        'Baseline_mm': float(individual_russia[model_name]['baseline'].mean()),
        'Future_mm': float(individual_russia[model_name]['future'].mean()),
        'Anomaly_mm': float(individual_russia[model_name]['anomaly'].mean()),
        'Change_percent': float(individual_russia[model_name]['anomaly'].mean()) / 
                         float(individual_russia[model_name]['baseline'].mean()) * 100
    })

# Add ensemble - Russia
summary_data.append({
    'Region': 'Russia',
    'Model': 'Ensemble Mean',
    'Type': 'Ensemble',
    'Baseline_mm': float(ensemble_russia['baseline'].mean()),
    'Future_mm': float(ensemble_russia['future'].mean()),
    'Anomaly_mm': float(ensemble_russia['anomaly'].mean()),
    'Change_percent': float(ensemble_russia['anomaly'].mean()) / 
                     float(ensemble_russia['baseline'].mean()) * 100
})

summary_df = pd.DataFrame(summary_data)
summary_df.to_csv('MultiModel_Ensemble_Summary.csv', index=False, float_format='%.2f')

print("="*100)
print("MULTI-MODEL ENSEMBLE SUMMARY")
print("="*100)
print(summary_df.to_string(index=False))
print("\n  Summary saved to: MultiModel_Ensemble_Summary.csv")

## Step 10: Export NetCDF Files

In [ ]:
# Export ensemble data
ensemble_indonesia['anomaly'].to_netcdf('Ensemble_Anomaly_Indonesia.nc')
ensemble_russia['anomaly'].to_netcdf('Ensemble_Anomaly_Russia.nc')

ensemble_indonesia['std'].to_netcdf('Ensemble_Uncertainty_Indonesia.nc')
ensemble_russia['std'].to_netcdf('Ensemble_Uncertainty_Russia.nc')

# Water stress: export both the classified categories AND the raw WSI %
water_stress_indonesia.to_netcdf('Ensemble_WaterStress_Indonesia.nc')
water_stress_russia.to_netcdf('Ensemble_WaterStress_Russia.nc')

indo_wsi.to_netcdf('Ensemble_WSI_Indonesia.nc')
russia_wsi.to_netcdf('Ensemble_WSI_Russia.nc')

print("NetCDF files exported:")
for f in ['Ensemble_Anomaly_Indonesia.nc', 'Ensemble_Anomaly_Russia.nc',
          'Ensemble_Uncertainty_Indonesia.nc', 'Ensemble_Uncertainty_Russia.nc',
          'Ensemble_WaterStress_Indonesia.nc', 'Ensemble_WaterStress_Russia.nc',
          'Ensemble_WSI_Indonesia.nc', 'Ensemble_WSI_Russia.nc']:
    print(f'  ✓ {f}')


## Step 11: Move Files to Output Directory

In [ ]:
current_dir = os.getcwd()
PROJECT_DIR = os.path.dirname(current_dir)
OUTPUT_DIR = os.path.join(PROJECT_DIR, 'data', 'multi_model_ensemble')

# Create output directory
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Current working directory: {current_dir}")
print(f"Output directory: {OUTPUT_DIR}")

# Find all output files
current_files = [f for f in os.listdir('.') if f.endswith(('.png', '.csv', '.nc')) and 
                ('Ensemble' in f or 'MultiModel' in f)]

if len(current_files) > 0:
    print(f"\nMoving {len(current_files)} files...")
    print("-" * 60)
    
    moved_count = 0
    for file in sorted(current_files):
        try:
            source = os.path.join(current_dir, file)
            destination = os.path.join(OUTPUT_DIR, file)
            shutil.move(source, destination)
            file_size = os.path.getsize(destination) / 1024
            print(f"  {file} ({file_size:.1f} KB)")
            moved_count += 1
        except Exception as e:
            print(f"✗ {file} - Error: {e}")
    
    print("-" * 60)
    print(f"\n  Successfully moved {moved_count}/{len(current_files)} files!")
    print(f"\nFiles are now in: {OUTPUT_DIR}")
else:
    print("\nNo ensemble output files found.")

print("\n" + "="*55)
print("    MULTI-MODEL ENSEMBLE ANALYSIS COMPLETE!    ")
print("="*55)